# Candidate diagnostics reproducibility notebook

This notebook reruns the deterministic, offline archive audit and checks its controlling totals. It makes no network or LLM calls. The archived final correctness remains authoritative; current-code recomputation is sensitivity-only. Cells were validated top-to-bottom with the same Python interpreter through a standard-library runner because this workspace venv does not include Jupyter.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

output_dir = Path.cwd()
script = output_dir / 'analyze_candidate.py'
completed = subprocess.run(
    [sys.executable, str(script)],
    cwd=output_dir,
    check=True,
    capture_output=True,
    text=True,
)
print(completed.stdout)

{
  "status": "ok",
  "case_count": 100,
  "pre_correct": {"count": 80, "denominator": 100, "rate": 0.8, "percent": 80.0},
  "final_correct": {"count": 36, "denominator": 100, "rate": 0.36, "percent": 36.0},
  "stage_loss": {"count": 45, "denominator": 100, "rate": 0.45, "percent": 45.0}
}


In [2]:
summary = json.loads((output_dir / 'summary.json').read_text(encoding='utf-8'))
overall = summary['groups']['overall']
control = {
    'case_count': overall['case_count'],
    'pre_correct': overall['pre_correct']['count'],
    'final_correct': overall['final_correct']['count'],
    'stage_loss': overall['flags']['pre_correct_final_noncorrect']['count'],
    'primary_attribution': {key: value['count'] for key, value in overall['primary_attribution'].items()},
}
assert control == {
    'case_count': 100,
    'pre_correct': 80,
    'final_correct': 36,
    'stage_loss': 45,
    'primary_attribution': {
        'generation_or_reasoning_error': 17,
        'no_observed_semantic_failure': 36,
        'post_generation_grounding_damage': 45,
        'retrieval_noise_likely': 2,
    },
}
control

{'case_count': 100, 'pre_correct': 80, 'final_correct': 36, 'stage_loss': 45, 'primary_attribution': {'generation_or_reasoning_error': 17, 'no_observed_semantic_failure': 36, 'post_generation_grounding_damage': 45, 'retrieval_noise_likely': 2}}

In [3]:
freshness = summary['freshness_audit']
expected_loss = summary['expected_terms_full_then_lost']
assert [item['case_id'] for item in freshness['cases']] == ['P3500-EASY-011', 'P3500-MEDIUM-047']
assert expected_loss['count'] == 43
{
    'freshness_gate_only_cases': [item['case_id'] for item in freshness['cases']],
    'expected_terms_full_then_lost': expected_loss['count'],
    'expected_loss_by_primary': expected_loss['by_primary_attribution'],
    'data_quality_gate': summary['data_quality_gate']['status'],
}

{'freshness_gate_only_cases': ['P3500-EASY-011', 'P3500-MEDIUM-047'], 'expected_terms_full_then_lost': 43, 'expected_loss_by_primary': {'generation_or_reasoning_error': 3, 'no_observed_semantic_failure': 1, 'post_generation_grounding_damage': 39}, 'data_quality_gate': 'passed'}